In [10]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time

In [50]:
prod_url = pd.read_csv('prod_urls.csv')

In [51]:
prod_url = prod_url.drop(columns=['status', 'discovered_at'])
prod_url['product_id'] = prod_url['product_id'].astype(int)
prod_url.head(5)

,product_id,url
0,698859,https://www.naiin.com/product/detail/698859/
1,698857,https://www.naiin.com/product/detail/698857/
2,698856,https://www.naiin.com/product/detail/698856/
3,699421,https://www.naiin.com/product/detail/699421/
4,698858,https://www.naiin.com/product/detail/698858/


In [53]:
examples = prod_url.tail(5)
examples

,product_id,url
186961,134055,https://www.naiin.com/product/detail/134055/
186962,152183,https://www.naiin.com/product/detail/152183/
186963,125673,https://www.naiin.com/product/detail/125673/
186964,25466,https://www.naiin.com/product/detail/25466/
186965,23362,https://www.naiin.com/product/detail/23362/


In [ ]:
import re
import json
import html

def extract_extra_info(html_content):
    """เวอร์ชันที่รับเป็น String (response.text) และใช้ Regex หาข้อมูล"""
    extra_data = {}
    try:
        # ค้นหาข้อความที่ขึ้นต้นด้วย AverageRating ไปจนถึงปีกกาปิด }
        # ใช้ re.search เพื่อหา Pattern ใน String ตรงๆ
        match = re.search(r'AverageRating&quot;:(.+?)\}', html_content)
        
        if match:
            # นำข้อมูลมาประกอบร่างเป็น JSON string ที่สมบูรณ์
            raw_json = '{"AverageRating":' + match.group(1) + '}'
            # แปลง &quot; กลับเป็นเครื่องหมายคำพูด "
            clean_json = html.unescape(raw_json)
            # แปลงเป็น Python Dictionary
            json_obj = json.loads(clean_json)
            
            # ดึงค่าที่ต้องการ
            extra_data = {
                'AverageRating': json_obj.get('AverageRating') or None,
                'TotalRating': json_obj.get('TotalRating') or None,
                'NumberOfPage': json_obj.get('NumberOfPage') or None,
                'Width': json_obj.get('Width') or None,
                'Height': json_obj.get('Height') or None,
                'Thick': json_obj.get('Thick') or None,
                'GrossWeightKG': json_obj.get('GrossWeight') or None,
                'FileSize': json_obj.get('FileSizeMB') or None
            }
    except Exception as e:
        # ถ้าพัง ให้คืนค่าว่างไปก่อน สคริปต์หลักจะได้เดินหน้าต่อได้
        print(f"  [Warning] Extra info skip: {e}")
        
    return extra_data

def scrape(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.naiin.com/',
    }
    
    time.sleep(1.5) # เพิ่มเวลาหน่อยกันโดนบล็อก
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, 'lxml')
    
    # ข้อมูลพื้นฐานจาก Metadata
    data = {'url': url}
    meta_map = {
        'Title': soup.find('meta', property='og:title'),
        'Price_Full': soup.find('meta', property='og:product:price:amount'),
        'Price_Sale': soup.find('meta', property='og:product:sale_price:amount'),
        'Barcode': soup.find('meta', property='book:isbn'),
        'Release_Date': soup.find('meta', property='book:release_date'),
        'keywords': soup.find('meta', attrs={'name': 'keywords'}),
    }

    for key, tag in meta_map.items():
        data[key] = tag['content'] if tag else "N/A"

    # --- ส่วนที่แก้ปัญหา Error ---
    # ส่ง response.text (String) เข้าไป
    extra_info = extract_extra_info(response.text)
    
    # รวมข้อมูลเข้าด้วยกัน
    data.update(extra_info)
    
    return data

In [54]:
import time

# 1. บันทึกเวลาเริ่มต้น
start_time = time.time()
print(f"🚀 Started at: {time.strftime('%H:%M:%S', time.localtime(start_time))}")

test_results = []

# ทดสอบกับตัวอย่าง (สมมติเอา 5 แถวแรก)
for _, row in examples.head(5).iterrows():
    print(f"Scraping ID: {row['product_id']} | URL: {row['url']}")
    
    # รันฟังก์ชัน Scrape
    res = scrape(row['url'])
    
    # แทรก product_id ไว้ที่ตำแหน่งแรก
    final_row = {'product_id': row['product_id']}
    final_row.update(res)
    
    test_results.append(final_row)

# 2. บันทึกเวลาสิ้นสุด
end_time = time.time()
duration = end_time - start_time # วินาที

print("-" * 30)
print(f"🏁 Finished at: {time.strftime('%H:%M:%S', time.localtime(end_time))}")
print(f"⏱️ Total Duration: {duration:.2f} seconds")

# 3. คำนวณค่าเฉลี่ย (ช่วยในการประเมินเวลาสำหรับงานแสนรายการ)
avg_time = duration / len(test_results)
print(f"📊 Average time per item: {avg_time:.2f} seconds")

🚀 Started at: 17:32:01
Scraping ID: 134055 | URL: https://www.naiin.com/product/detail/134055/
Scraping ID: 152183 | URL: https://www.naiin.com/product/detail/152183/
Scraping ID: 125673 | URL: https://www.naiin.com/product/detail/125673/
Scraping ID: 25466 | URL: https://www.naiin.com/product/detail/25466/
Scraping ID: 23362 | URL: https://www.naiin.com/product/detail/23362/
------------------------------
🏁 Finished at: 17:32:14
⏱️ Total Duration: 13.12 seconds
📊 Average time per item: 2.62 seconds


In [55]:
test_results

[{'product_id': 134055,
  'url': 'https://www.naiin.com/product/detail/134055/',
  'Title': 'วารสาร สารวุฒิสภา ฉ.มกราคม 57 (ฟรี)E-Magazines | e-magazine ร้านหนังสือนายอินทร์',
  'Price_Full': 'N/A',
  'Price_Sale': 'N/A',
  'Barcode': '9000024275',
  'Release_Date': '2014-01-01',
  'keywords': 'วารสาร สารวุฒิสภา ฉ.มกราคม 57 (ฟรี), สนง.เลขาธิการวุฒิสภา, สนง.เลขาธิการวุฒิสภา, , , ',
  'AverageRating': None,
  'TotalRating': None,
  'NumberOfPage': None,
  'Width': None,
  'Height': None,
  'Thick': None,
  'GrossWeightKG': None,
  'FileSize': None},
 {'product_id': 152183,
  'url': 'https://www.naiin.com/product/detail/152183/',
  'Title': 'A SCATTERED WORLDBooks | ร้านหนังสือนายอินทร์',
  'Price_Full': '195',
  'Price_Sale': '185.25',
  'Barcode': '9786167552491',
  'Release_Date': '2015-06-09',
  'keywords': 'A SCATTERED WORLD, Marcel Barang, PAJONPHAI PUBLISHING, หนังสือต่างประเทศ, หนังสือต่างประเทศ, WORLD,SCATTERED',
  'AverageRating': None,
  'TotalRating': None,
  'NumberOfPage': 1

In [56]:
pd.DataFrame(test_results)

,product_id,url,Title,Price_Full,Price_Sale,Barcode,Release_Date,keywords,AverageRating,TotalRating,NumberOfPage,Width,Height,Thick,GrossWeightKG,FileSize
0,134055,https://www.naiin.com/product/detail/134055/,วารสาร สารวุฒิสภา ฉ.มกราคม 57 (ฟรี)E-Magazines...,N/A,N/A,9000024275,2014-01-01,"วารสาร สารวุฒิสภา ฉ.มกราคม 57 (ฟรี), สนง.เลขาธ...",None,None,NaN,NaN,NaN,NaN,NaN,None
1,152183,https://www.naiin.com/product/detail/152183/,A SCATTERED WORLDBooks | ร้านหนังสือนายอินทร์,195,185.25,9786167552491,2015-06-09,"A SCATTERED WORLD, Marcel Barang, PAJONPHAI PU...",None,None,144.0,10.5,17.9,0.9,0.110,None
2,125673,https://www.naiin.com/product/detail/125673/,คู่มือ+ข้อสอบนักวิชาการคลัง(กรมบัญชีกลางBooks ...,250,237.5,9786162462085,2015-01-22,"คู่มือ+ข้อสอบนักวิชาการคลัง(กรมบัญชีกลาง, สิงห...",None,None,303.0,29.2,20.5,1.9,0.710,None
3,25466,https://www.naiin.com/product/detail/25466/,Quilt ผ้าสไตล์ HawaiianBooks | ร้านหนังสือนายอ...,250,237.5,9786165301763,2014-12-12,"Quilt ผ้าสไตล์ Hawaiian, กองบรรณาธิการแม่บ้าน,...",None,None,NaN,26.0,20.8,0.8,0.465,None
4,23362,https://www.naiin.com/product/detail/23362/,ดอกไม้ผ้าแฮนด์เมดBooks | ร้านหนังสือนายอินทร์,250,237.5,9786165301770,2014-11-01,"ดอกไม้ผ้าแฮนด์เมด, กองบรรณาธิการ, ประดิดประดอย...",None,None,NaN,26.1,21.0,0.6,0.325,None
